In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
data = pd.read_csv("beef dataset.csv")
data.columns = data.columns.str.strip()   # ← IMPORTANT


# Target column
TARGET_COL = "class"

# Drop rows with missing class labels
data.dropna(subset=[TARGET_COL], inplace=True)

# Fill missing numeric sensor values
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].mean())

# Encode class labels
le = LabelEncoder()
data[TARGET_COL] = le.fit_transform(data[TARGET_COL])

print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# Features and target
X = data.drop(TARGET_COL, axis=1)
y = data[TARGET_COL]

LEAK_COLS = ["minute", "TVC"]
X = X.drop(columns=LEAK_COLS, errors="ignore")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=150,
    random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Save model
joblib.dump(model, "beef_spoilage_rf_model.pkl")
joblib.dump(le, "beef_label_encoder.pkl")

print("Model trained and saved successfully.")

Class mapping: {'acceptable': np.int64(0), 'excellent': np.int64(1), 'good': np.int64(2), 'spoiled': np.int64(3)}
Accuracy: 0.9978046103183315
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        37
           1       0.98      1.00      0.99        57
           2       1.00      0.98      0.99        42
           3       1.00      1.00      1.00       775

    accuracy                           1.00       911
   macro avg       1.00      0.99      0.99       911
weighted avg       1.00      1.00      1.00       911

Model trained and saved successfully.


In [3]:
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

MQ5            0.195178
MQ2            0.139422
MQ3            0.135740
MQ6            0.134098
MQ9            0.084042
MQ135          0.069070
MQ4            0.064909
MQ8            0.059613
MQ136          0.051185
Humidity       0.037112
Temperature    0.027013
MQ7            0.002617
dtype: float64


In [5]:
import pandas as pd
import joblib

model = joblib.load("beef_spoilage_rf_model.pkl")

features = model.feature_names_in_

values = {
    "MQ135": 27.01,
    "MQ136": 8.25,
    "MQ2": 8.87,
    "MQ3": 11.97,
    "MQ4": 4.97,
    "MQ5": 11.75,
    "MQ6": 12.51,
    "MQ7": 1.17,
    "MQ8": 14.5,
    "MQ9": 14.1,
    "Humidity": 73,
    "Temperature": 37
}

# Reorder automatically
sample = pd.DataFrame([[values[f] for f in features]], columns=features)

pred = model.predict(sample)

labels = {
    0: "Excellent",
    1: "Good",
    2: "Acceptable",
    3: "Spoiled"
}

print("Predicted status:", labels[pred[0]])

Predicted status: Excellent
